# Datamine FFUNC Process Example

This notebook demonstrates how to configure and run the **`ffunc`** process wrapper in `dmstudio`.

## Process Description

## FFUNC

The F function estimates the variance of a point in a block, based on a variogram model which is selected from the input variogram model file **VMODPARM**.

The block whose F value is to be calculated is simulated by a 3 dimensional matrix of discretisation points. The F value is calculated as the average value of the variogram between each possible pair of discretisation points.

The number of discretisation points in each of the X, Y and Z directions is specified by parameters **IPOINTS** , **JPOINTS** and **KPOINTS**. It is recommended that a minimum of 4 x 4 x 4 points is used. The processing speed is proportional to the number of points used.

You will be prompted interactively to enter the X, Y and Z dimensions of the block. The F value will be displayed in the Output window, and you may then enter the dimensions of another block, or ! to terminate the process.

By estimating the F-function for two block sizes, the theoretical Dispersion Variance for selective mining units within a kriged block can be calculated. This can then be used as input to processes [SMUHIS](<smuhis.md>) and [SMUMOD](<smumod.md>).

**Note** : You will be prompted interactively to enter the dimensions of the block whose F value is required.

### Input Files:

* **vmodparm** (Variogram - Model):
  Input variogram model file.
  Required=Yes

### Output Files:

* **out** (Undefined):
  Optional output file. This will contain the block dimensions and F value in fields
  **XINC** , **YINC** , **ZINC** and **FVALUE**.
  Required=No

### Fields:

### Parameters:

* **vmodnum**:
  Variogram model number, as defined by **VREFNUM** field in **VMODPARM** file. Default
  (1).
  Range=Undefined
  Values=Undefined
  Default=1
  Required=No

* **log**:
  Log/Normal variogram flag. Default(0). The variogram model, as defined by **VGRAM** ,
  is Normal if LOG =0 or Lognormal if LOG =1.
  Range=0,1
  Values=0,1
  Default=0
  Required=No

* **ipoints**:
  Number of discretisation points in X dimension to simulate block (6)
  Range=Undefined
  Values=Undefined
  Default=6
  Required=No

* **jpoints**:
  Number of discretisation points in Y dimension to simulate block (6)
  Range=Undefined
  Values=Undefined
  Default=6
  Required=No

* **kpoints**:
  Number of discretisation points in Z dimension to simulate block (6)
  Range=Undefined
  Values=Undefined
  Default=6
  Required=No

In [ ]:
# Step 1: Connect to Datamine and Initialize Sandbox
import os
import shutil
import glob

import pandas as pd

from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Initialize active project sandbox using the shared test_sandbox project
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()
agent.initialize_sandbox(notebook_folder)

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('ffunc')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine database locally to this sandbox folder using a `t_` prefix.

In [ ]:
# Step 3: Copy VBOP datasets dynamically from Database to test_sandbox
files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

agent.copy_database_files(files_to_copy)

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute ffunc
print("Running ffunc...")
dm_cmd.ffunc(
    vmodparm_i='t_vpar',  # required
    # out_o='t_ffunc_out',  # optional
    # vmodnum_p=1,  # optional
    # log_p=0,  # optional
    # ipoints_p=6,  # optional
    # jpoints_p=6,  # optional
    # kpoints_p=6,  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("ffunc execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = 't_ffunc_out.dmx'
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob("t_*.*")
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    if os.path.exists(f):
        try:
            os.remove(f)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = '__pycache__'
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")